# AI-sfa 영상 → SFA 딥러닝 학습 — Kaggle GPU

수집 SFA 361개(4711_2 합작 28개는 최종 시험용으로 뺌)를 그려서 영상으로 인코딩하고, 영상 장마다 '선 가운데·점 자리(선 끝)·선 방향·칠한 면'을 맞히게 학습한다.
**커리큘럼(사용자 10-29):** 1080 → 720 → 480 → 360 순서로 어려워진다. 단계마다 새 해상도 70% + 앞 단계 30%.
저장소(공개)에서 파일·도구를 직접 받으므로 따로 올릴 것은 없다.

**설정:**
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On**
3. (선택) **Add-ons → Secrets**에 `RCLONE_CONFIG_GDRIVE_TOKEN`을 붙이면 결과가 Drive `AI-sfa/02_분석자료/vec/`로 올라간다. 없으면 Output 탭에서 `vec_out.zip`을 받는다.

**GPU 시간이 없을 때:** Accelerator를 None으로 두고 실행하면 자료 만들기와 짧은 시험까지만 하고 멈춘다(오류 확인용).

**실행:** 오른쪽 위 **Save Version → Save & Run All (Commit)** — 브라우저를 닫아도 돈다.
예상: 자료 만들기 20–40분 + 짧은 시험 몇 분 + 본 학습 4시간 ≈ 5시간(추정).
결과 `vec_out.zip`: `best.pt`(최종 단계에서 검증이 가장 좋은 모델), `last.pt`, 단계별 `stage*.pt`, `log.jsonl`(손실 기록).

In [ ]:
# 1. (선택) Drive 연결 — 비밀값이 없으면 건너뛴다. 인증값은 출력하지 않는다
import os, json, base64, pathlib, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    sc = UserSecretsClient()
    try:  # 사용자 Claude 환경 변수와 같은 이름: rclone 토큰 JSON 그대로(10-28)
        d = {'token': sc.get_secret('RCLONE_CONFIG_GDRIVE_TOKEN').strip()}
        json.loads(d['token'])
    except Exception:  # 예전 방식: client_id·secret·token을 담은 base64 JSON
        raw = sc.get_secret('RCLONE_GDRIVE_AUTH_B64').strip()
        d = json.loads(base64.b64decode(raw + '=' * (-len(raw) % 4)))
    p = pathlib.Path.home() / '.config/rclone'; p.mkdir(parents=True, exist_ok=True)
    lines = ['[gdrive]', 'type = drive', 'scope = drive.readonly,drive.file']
    if d.get('client_id'): lines.append('client_id = ' + d['client_id'])
    if d.get('client_secret'): lines.append('client_secret = ' + d['client_secret'])
    lines.append('token = ' + d['token'])
    (p / 'rclone.conf').write_text('\n'.join(lines) + '\n'); os.chmod(p / 'rclone.conf', 0o600)
    del d, lines
    subprocess.run('cd /tmp && curl -sSLo rc.zip https://downloads.rclone.org/rclone-current-linux-amd64.zip && unzip -qo rc.zip && cp rclone-*/rclone /usr/local/bin/', shell=True, check=True)
    print('Drive 연결됨')
except Exception as e:
    print('Drive 연결 없이 진행(결과는 Output 탭):', type(e).__name__)

In [ ]:
# 2. 작업자 파일 쓰기 → 실행(설치 → 저장소 받기 → 해상도별 자료 → 짧은 시험 → 본 학습 → zip)
open('/tmp/kaggle_vec_worker.py', 'w', encoding='utf-8').write("#!/usr/bin/env python3\n\"\"\"Kaggle 작업자: 영상 → SFA 딥러닝(자료 만들기 → 짧은 시험 → 본 학습 → 결과 zip·Drive) (2026-10-29).\n저장소(공개)에서 SFA 파일과 도구를 받는다. 4711_2 합작 28개는 학습에서 빠진다(vec_data.HOLDOUT, 최종 시험용).\n\"\"\"\nimport argparse\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport time\n\n\ndef sh(cmd):\n    print('$', cmd, flush=True)\n    subprocess.run(cmd, shell=True, check=True)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--branch', default='ccr-bccf15a6-d6ee46')\n    ap.add_argument('--hours', type=float, default=4.0)\n    ap.add_argument('--repeat', type=int, default=2)\n    ap.add_argument('--max-frames', type=int, default=24)\n    ap.add_argument('--out', default='/kaggle/working')\n    a = ap.parse_args()\n    t0 = time.time()\n    sh('pip install -q skia-python')\n    if not os.path.isdir('/kaggle/tmp/AI-sfa'):\n        sh(f'git clone -q --depth 1 -b {a.branch} https://github.com/twoapple08/AI-sfa /kaggle/tmp/AI-sfa')\n    tools = '/kaggle/tmp/AI-sfa/tools'\n    data = '/kaggle/tmp/vd'\n    cpus = os.cpu_count() or 4\n    def gen(res):\n        d = os.path.join(data, res)\n        if os.path.isdir(d) and len(os.listdir(d)) > 50:\n            return\n        sh(f'cd {tools} && python -u vec_data.py {d} --res {res} --repeat {a.repeat} --max-frames {a.max_frames} --workers {cpus} --seed {res}')\n    out = os.path.join(a.out, 'vec_out')\n    # 1080 자료만 먼저 → 짧은 시험(문법·GPU 메모리; 서버에 torch가 없어 여기서 처음 돈다) → 나머지 자료 → 본 학습\n    gen('1920')\n    sh(f'cd {tools} && python -u vec_train.py --data {data} --out /kaggle/tmp/smoke --smoke --workers 2')\n    for res in ('1280', '854', '640'):  # 커리큘럼 단계별 자료(사용자 10-29)\n        gen(res)\n    sh(f'du -sh {data}/*')\n    print(f'자료 만들기·시험 {(time.time() - t0) / 60:.1f}분', flush=True)\n    import torch\n    if not torch.cuda.is_available():  # GPU 시간이 없을 때: 자료·짧은 시험까지만 확인하고 멈춤(사용자 10-29, Kaggle GPU 0시간)\n        print('GPU 없음 → 자료 만들기와 짧은 시험까지만 확인하고 멈춥니다. 오류 없이 여기까지 왔으면 GPU 시간이 돌아왔을 때 그대로 다시 실행하면 됩니다.', flush=True)\n        return\n    sh(f'cd {tools} && python -u vec_train.py --data {data} --out {out} --hours {a.hours} --workers {max(2, cpus - 1)}')\n    shutil.copy(os.path.join(tools, 'vec_train.py'), out)\n    sh(f'cd {a.out} && zip -qr vec_out.zip vec_out')\n    if shutil.which('rclone'):\n        try:\n            sh(f'rclone copy {a.out}/vec_out.zip \"gdrive:AI-sfa/02_분석자료/vec/\" -P')\n        except Exception as e:\n            print('Drive 올리기 실패(Output 탭에서 받기):', e)\n    print(f'전체 {(time.time() - t0) / 60:.1f}분', flush=True)\n\n\nif __name__ == '__main__':\n    main()\n")
!python -u /tmp/kaggle_vec_worker.py --hours 4